# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jazaalee/FlyRank-StarterNotebook/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*
Unit of analysis + time window

One row in this table is **one page, on one specific day**, for one client. So if the same page shows up on 5 different days, that's 5 separate rows, not one. I'm looking at March 2026 specifically (2026-03-01 to 2026-03-31), which is just a normal middle-of-the-panel month, not the very last month in the dataset. I'm avoiding the last month on purpose, since that's the "sealed" window that later assignments will actually try to predict, so using it now to build my label logic would basically mean peeking at the answer early.

 I grouped the data by page, client, and date together and looked for any group that showed up more than once. Nothing came back, which means every single row really is unique at that level, exactly what I claimed. I also checked the date range directly and confirmed it runs cleanly from March 1st to March 31st, with nothing missing or spilling into other months.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*
Fields: feature / label / context / excluded

Before touching anything, I sorted every column I might use into one of four buckets, since mixing these up is exactly how leakage sneaks in later.

**Feature (things the model is allowed to learn from):** gsc_impressions, gsc_clicks, gsc_sum_position, and the AI-referral columns like sessions_ai. These are all things that were genuinely true and knowable on the day they were recorded, before anyone made any decision about the page.

**Label or proxy (the thing I'm trying to predict):** whether a page's impressions dropped from the first half of March to the second half. I'm calling this a proxy on purpose, not a real outcome, because it's just comparing two halves of the same month I already have, not something that happened after a decision point.

**Context (used only to organize the data, never fed to a model)**: report_date, client_hash_id, and content_hash_id. These are just identifiers and dates, they don't describe the page's performance, so a model shouldn't be learning patterns from them directly, only using them to group and join data correctly.

**Excluded (things I'm deliberately not touching, and why)**: fact_content_query_90d, the table with query-level breakdowns. I'm leaving it out because my question is about how a single page trends over time, not about which specific search terms bring it traffic, so pulling that table in would add complexity and a new leakage risk I don't actually need for this task.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*
Verify it with queries

Every claim I made above has a query backing it up, not just my word for it.

Grain: I grouped the data by page, client, and date, and checked if any combination showed up more than once. It came back empty, zero rows, which proves the grain really is what I said: one row per page, per client, per day, with no duplicates hiding in there.

Counts and window: I counted the total rows and checked the earliest and latest date in the table. It came back as 9,841,378 rows, running exactly from 2026-03-01 to 2026-03-31. That matches what I expected for a full month with nothing missing at either end.

Missing values (availability): I checked how many rows actually have real search data versus real analytics data, using gsc_data_available IS TRUE and ga4_data_available IS TRUE. Out of the 9.8 million rows, only about 36.7% have GSC (search) data available, and only about 4.2% have GA4 (analytics) data available. This isn't random, it lines up with something the lane guide already warned about: not every client has GA4 tracking hooked up, and GSC data itself only shows up on days a page actually got some visibility. So if I build a feature using GA4 data (like sessions or engagement), I'm automatically working with a much smaller, specific slice of pages, not the whole dataset, and I need to say that clearly instead of pretending every feature covers the same rows.

In [1]:
from google.colab import userdata
import os

os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
print("Token loaded:", bool(os.environ.get("HF_TOKEN")))


Token loaded: True


In [2]:
from huggingface_hub import HfApi

api = HfApi()
files = api.list_repo_files("FlyRank/internship-warehouse", repo_type="dataset")
for f in sorted(files)[:80]:
    print(f)

print("\nTotal files:", len(files))

.gitattributes
README.md
dim_clients.parquet
dim_content.parquet
fact_content_daily_performance/month=2025-01/data_0.parquet
fact_content_daily_performance/month=2025-02/data_0.parquet
fact_content_daily_performance/month=2025-03/data_0.parquet
fact_content_daily_performance/month=2025-04/data_0.parquet
fact_content_daily_performance/month=2025-05/data_0.parquet
fact_content_daily_performance/month=2025-06/data_0.parquet
fact_content_daily_performance/month=2025-07/data_0.parquet
fact_content_daily_performance/month=2025-08/data_0.parquet
fact_content_daily_performance/month=2025-09/data_0.parquet
fact_content_daily_performance/month=2025-10/data_0.parquet
fact_content_daily_performance/month=2025-11/data_0.parquet
fact_content_daily_performance/month=2025-12/data_0.parquet
fact_content_daily_performance/month=2026-01/data_0.parquet
fact_content_daily_performance/month=2026-02/data_0.parquet
fact_content_daily_performance/month=2026-03/data_0.parquet
fact_content_daily_performance/mont

In [3]:
!pip install -q duckdb

import duckdb

con = duckdb.connect()
con.sql("INSTALL httpfs; LOAD httpfs;")
con.sql("CREATE OR REPLACE SECRET hf_token (TYPE huggingface, TOKEN '" + os.environ["HF_TOKEN"] + "');")

print("DuckDB ready")

DuckDB ready


In [4]:
march_path = "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/data_0.parquet"

con.sql(f"""
    CREATE OR REPLACE VIEW march_daily AS
    SELECT * FROM read_parquet('{march_path}')
""")

con.sql("SELECT COUNT(*) AS row_count FROM march_daily").show()
con.sql("DESCRIBE march_daily").show()

┌───────────┐
│ row_count │
│   int64   │
├───────────┤
│   9841378 │
└───────────┘

┌────────────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│    column_name     │ column_type │  null   │   key   │ default │  extra  │
│      varchar       │   varchar   │ varchar │ varchar │ varchar │ varchar │
├────────────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ report_date        │ DATE        │ YES     │ NULL    │ NULL    │ NULL    │
│ client_hash_id     │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ content_hash_id    │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ client_has_gsc     │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ client_has_ga4     │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ gsc_data_available │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ ga4_data_available │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ gsc_impressions    │ BIGINT      │ YES     │ NULL    │ NULL    │ N

In [5]:
grain_check = con.sql("""
    SELECT report_date, client_hash_id, content_hash_id, COUNT(*) AS c
    FROM march_daily
    GROUP BY report_date, client_hash_id, content_hash_id
    HAVING c > 1
    LIMIT 5
""")
grain_check.show()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌─────────────┬────────────────┬─────────────────┬───────┐
│ report_date │ client_hash_id │ content_hash_id │   c   │
│    date     │    varchar     │     varchar     │ int64 │
├─────────────┴────────────────┴─────────────────┴───────┤
│                         0 rows                         │
└────────────────────────────────────────────────────────┘



In [6]:
con.sql("""
    SELECT COUNT(*) AS row_count,
           MIN(report_date) AS earliest_date,
           MAX(report_date) AS latest_date
    FROM march_daily
""").show()

┌───────────┬───────────────┬─────────────┐
│ row_count │ earliest_date │ latest_date │
│   int64   │     date      │    date     │
├───────────┼───────────────┼─────────────┤
│   9841378 │ 2026-03-01    │ 2026-03-31  │
└───────────┴───────────────┴─────────────┘



In [7]:
con.sql("""
    SELECT
        COUNT(*) AS total_rows,
        SUM(CASE WHEN gsc_data_available IS TRUE THEN 1 ELSE 0 END) AS gsc_available_rows,
        SUM(CASE WHEN ga4_data_available IS TRUE THEN 1 ELSE 0 END) AS ga4_available_rows
    FROM march_daily
""").show()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌────────────┬────────────────────┬────────────────────┐
│ total_rows │ gsc_available_rows │ ga4_available_rows │
│   int64    │       int128       │       int128       │
├────────────┼────────────────────┼────────────────────┤
│    9841378 │            3611061 │             413966 │
└────────────┴────────────────────┴────────────────────┘



In [8]:
first_half = con.sql("""
    SELECT
        content_hash_id,
        client_hash_id,
        SUM(gsc_impressions) AS impressions_first_half,
        SUM(gsc_clicks) AS clicks_first_half,
        SUM(gsc_sum_position) AS sum_position_first_half,
        COUNT(DISTINCT report_date) AS days_with_data_first_half,
        SUM(scroll_events) AS scroll_events_first_half
    FROM march_daily
    WHERE report_date <= DATE '2026-03-15'
    GROUP BY content_hash_id, client_hash_id
""")

first_half.show()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌──────────────────────────┬─────────────────────────┬────────────────────────┬───────────────────┬─────────────────────────┬───────────────────────────┬──────────────────────────┐
│     content_hash_id      │     client_hash_id      │ impressions_first_half │ clicks_first_half │ sum_position_first_half │ days_with_data_first_half │ scroll_events_first_half │
│         varchar          │         varchar         │         int128         │      int128       │         int128          │           int64           │          int128          │
├──────────────────────────┼─────────────────────────┼────────────────────────┼───────────────────┼─────────────────────────┼───────────────────────────┼──────────────────────────┤
│ content_67741cce996cfafa │ client_62f4a7e64f5e0096 │                     38 │                 1 │                     170 │                        15 │                     NULL │
│ content_2e6360ad20fd7107 │ client_62f4a7e64f5e0096 │                    219 │                

In [9]:
second_half = con.sql("""
    SELECT
        content_hash_id,
        client_hash_id,
        SUM(gsc_impressions) AS impressions_second_half
    FROM march_daily
    WHERE report_date > DATE '2026-03-15'
    GROUP BY content_hash_id, client_hash_id
""")

second_half.show()


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌──────────────────────────┬─────────────────────────┬─────────────────────────┐
│     content_hash_id      │     client_hash_id      │ impressions_second_half │
│         varchar          │         varchar         │         int128          │
├──────────────────────────┼─────────────────────────┼─────────────────────────┤
│ content_73f21e612565035a │ client_3ffa76342f366962 │                       0 │
│ content_5a5be514ff559598 │ client_3ffa76342f366962 │                       0 │
│ content_05b377d0c8a5cfd8 │ client_3ffa76342f366962 │                       0 │
│ content_859b5acb04908ae3 │ client_3ffa76342f366962 │                       0 │
│ content_be99356ea2fc1df1 │ client_3ffa76342f366962 │                       1 │
│ content_11c9253606b7d3bd │ client_3ffa76342f366962 │                       0 │
│ content_42e4dc3c4026a190 │ client_3ffa76342f366962 │                       0 │
│ content_d86e1c84849226ba │ client_3ffa76342f366962 │                       0 │
│ content_0ef4614ed317d773 │

# **Building the five features**
I joined the first-half and second-half data together, keeping only pages that had at least some visibility in the first half (a page with zero impressions can't meaningfully decline). The label compares second-half impressions against first-half impressions. Every feature below comes only from the first half of March, before the point I'm comparing against, so nothing here can accidentally leak information from the window I'm trying to predict.

In [10]:
lane_frame = con.sql("""
    SELECT
        f.content_hash_id,
        f.client_hash_id,
        f.impressions_first_half,
        f.clicks_first_half,
        f.sum_position_first_half,
        f.days_with_data_first_half,
        f.scroll_events_first_half,
        COALESCE(s.impressions_second_half, 0) AS impressions_second_half
    FROM first_half f
    LEFT JOIN second_half s
        ON f.content_hash_id = s.content_hash_id
       AND f.client_hash_id = s.client_hash_id
    WHERE f.impressions_first_half > 0
""").df()

print(lane_frame.shape)
lane_frame.head()


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

(151981, 8)


,content_hash_id,client_hash_id,impressions_first_half,clicks_first_half,sum_position_first_half,days_with_data_first_half,scroll_events_first_half,impressions_second_half
0,content_dd83cb75985afc9c,client_3ffa76342f366962,1.0,0.0,8.0,13,0.0,1.0
1,content_bcfcf9f17422d40f,client_3ffa76342f366962,1.0,0.0,9.0,13,0.0,0.0
2,content_5e6c802f5491cae4,client_3ffa76342f366962,1.0,0.0,9.0,13,0.0,1.0
3,content_46eec13758f48ef3,client_3ffa76342f366962,3.0,0.0,2.0,13,0.0,1.0
4,content_982028f29ae0e58d,client_3ffa76342f366962,5.0,0.0,188.0,13,0.0,4.0


# **Five features, each knowable at the decision moment**

impressions_first_half — knowable because it's a direct count of what already happened by the midpoint of the month; nothing here depends on anything after that point.


avg_position_first_half — knowable because it's calculated purely from first-half data (sum_position / impressions); it reflects where the page was ranking before the decision point, not after.


ctr_first_half — knowable for the same reason as position: it's clicks / impressions computed only from the first 15 days, so it's a snapshot of behavior that already occurred.


days_with_data_first_half — knowable because it's just a count of how many days in the first half actually had any tracked activity; this also doubles as a data-quality signal (a page with only 2-3 days of data is less trustworthy than one with all 15).


scroll_events_first_half — knowable because it's summed only from the first-half window; I do note that this one has real missingness (some clients show NULL), so I'm treating it as a weaker, secondary feature rather than a core one.

In [11]:
import numpy as np

lane_frame["avg_position_first_half"] = (
    lane_frame["sum_position_first_half"] / lane_frame["impressions_first_half"]
)
lane_frame["ctr_first_half"] = (
    lane_frame["clicks_first_half"] / lane_frame["impressions_first_half"]
)

# The label: did impressions drop from first half to second half?
lane_frame["is_declining_proxy"] = (
    lane_frame["impressions_second_half"] < lane_frame["impressions_first_half"]
).astype(int)

features = [
    "impressions_first_half",
    "avg_position_first_half",
    "ctr_first_half",
    "days_with_data_first_half",
    "scroll_events_first_half",
]

lane_frame[features + ["is_declining_proxy"]].head(10)

,impressions_first_half,avg_position_first_half,ctr_first_half,days_with_data_first_half,scroll_events_first_half,is_declining_proxy
0,1.0,8.000000,0.0,13,0.0,0
1,1.0,9.000000,0.0,13,0.0,1
2,1.0,9.000000,0.0,13,0.0,0
3,3.0,0.666667,0.0,13,0.0,1
4,5.0,37.600000,0.0,13,0.0,1
5,1.0,9.000000,0.0,13,0.0,0
6,1.0,2.000000,0.0,13,1.0,1
7,1.0,1.000000,0.0,13,0.0,0
8,1.0,0.000000,0.0,13,0.0,0
9,3.0,2.333333,0.0,13,0.0,0


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*
The leakage trap

I deliberately added impressions_second_half, the exact column my label was built from, as a feature, just to see what happens.

Without it, my honest model using only the five legitimate first-half features got a ROC AUC of 0.526, barely better than a coin flip. That's a realistic, if humbling, number: it tells me these five features alone aren't strongly separating declining pages from stable ones yet, at least not with a simple logistic regression.

With impressions_second_half added in, the score jumped to a perfect 1.000. That's not a real result, it's the model literally being handed the answer. Since the label is "did second-half impressions drop below first-half impressions," giving the model second-half impressions directly means it can compute the label itself instead of learning any real pattern. A perfect score should be a red flag, not something to celebrate, real-world signals are almost never this clean.

I removed that column and kept the honest 0.526 as my real number. It's not an impressive score, but it's a true one, and it tells me I likely need better features, more history, or a different modeling approach before this becomes a strong lane, not that I should chase a suspiciously perfect result.

In [12]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

X_honest = lane_frame[features].fillna(0)
y = lane_frame["is_declining_proxy"]

X_train, X_test, y_train, y_test = train_test_split(X_honest, y, test_size=0.25, random_state=42)

model_honest = LogisticRegression(max_iter=1000, class_weight="balanced")
model_honest.fit(X_train, y_train)

honest_score = roc_auc_score(y_test, model_honest.predict_proba(X_test)[:, 1])
print(f"Honest ROC AUC (5 legitimate features only): {honest_score:.3f}")


Honest ROC AUC (5 legitimate features only): 0.526


In [13]:
X_leaky = lane_frame[features + ["impressions_second_half"]].fillna(0)

X_train_l, X_test_l, y_train_l, y_test_l = train_test_split(X_leaky, y, test_size=0.25, random_state=42)

model_leaky = LogisticRegression(max_iter=1000, class_weight="balanced")
model_leaky.fit(X_train_l, y_train_l)

leaky_score = roc_auc_score(y_test_l, model_leaky.predict_proba(X_test_l)[:, 1])
print(f"'Leaky' ROC AUC (with impressions_second_half included): {leaky_score:.3f}")

'Leaky' ROC AUC (with impressions_second_half included): 1.000


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.